In [1]:
import hydra
from lmetk.data.datasets import Dataset_ETT_hour


%reload_ext autoreload
%autoreload 2
%matplotlib inline


In [2]:
data = Dataset_ETT_hour(data_dir='datasets/ETT-small')

#/home/levtel/projects/TabPHM/datasets/ETT-small/ETTh1.csv

In [3]:
datasets = ['ETTh1']

for data_name in datasets:
#data_name = 'ETTh1'
    for flag in ['train', 'val', 'test']:
        with hydra.initialize(
                version_base="1.3",
                config_path='configs',
                job_name="run"
            ):
            #print('Current config file: ', config_file)
            parameters = hydra.compose(
                config_name="run.yaml",
                overrides=[f'dataset={data_name}', f'dataset.loader.flag={flag}'], 
                return_hydra_config=True, 
            )
            dataset_loader1 = hydra.utils.instantiate(parameters.dataset.loader)
            
    
            print(f"{flag} dataset size", dataset_loader1.__len__())

train dataset size 8161
val dataset size 2785
test dataset size 2785


### Behaviour of not refactored dataset

In [4]:
with hydra.initialize(
        version_base="1.3",
        config_path='configs',
        job_name="run"
    ):
    #print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f'dataset=ETTh1', f'dataset.loader.flag=val'], 
        return_hydra_config=True, 
    )
    dataset_loader1 = hydra.utils.instantiate(parameters.dataset.loader)

for i in range(2783, 2787):
    seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_loader1.__getitem__(i)
    print('Index', i)
    print(f"seq_x shape: {seq_x.shape}, seq_y shape: {seq_y.shape}, seq_x_mark shape: {seq_x_mark.shape}, seq_y_mark shape: {seq_y_mark.shape}")



Index 2783
seq_x shape: (384, 1), seq_y shape: (192, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (192, 4)
Index 2784
seq_x shape: (384, 1), seq_y shape: (192, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (192, 4)
Index 2785
seq_x shape: (384, 1), seq_y shape: (191, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (191, 4)
Index 2786
seq_x shape: (384, 1), seq_y shape: (190, 1), seq_x_mark shape: (384, 4), seq_y_mark shape: (190, 4)


### Refactored version

In [5]:

from lmetk.data.datasets import TimeSeriesDataset

data_name = 'ETTh1_refactored'

with hydra.initialize(
        version_base="1.3",
        config_path='configs',
        job_name="run"
    ):
    #print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f'dataset={data_name}'], 
        return_hydra_config=True, 
    )
    dataset_loader2 = hydra.utils.instantiate(parameters.dataset.loader)


# Get params of splitting and sliding
data_splitter_params = parameters.dataset.loader.data_splitter
seq_len = data_splitter_params.seq_len
label_len = data_splitter_params.label_len
pred_len = data_splitter_params.pred_len

# get data
splitted_data, splitted_timestamps = dataset_loader2.get_data()

dataset_train = TimeSeriesDataset(data=splitted_data['train'], 
                  time_features=splitted_timestamps['train'], 
                  seq_len=seq_len, 
                  label_len=label_len, 
                  pred_len=pred_len)

dataset_val = TimeSeriesDataset(data=splitted_data['val'],
                    time_features=splitted_timestamps['val'], 
                    seq_len=seq_len, 
                    label_len=label_len, 
                    pred_len=pred_len)
dataset_test = TimeSeriesDataset(data=splitted_data['test'],
                    time_features=splitted_timestamps['test'], 
                    seq_len=seq_len, 
                    label_len=label_len, 
                    pred_len=pred_len)
print(f"Train dataset size: {dataset_train.__len__()}")
print(f"Validation dataset size: {dataset_val.__len__()}")
print(f"Test dataset size: {dataset_test.__len__()}")

InstantiationException: Error in call to target '__init__.ETTh_loader.Dataset_ETT_hour_refactored':
TypeError("time_features() got an unexpected keyword argument 'use_features'")
full_key: dataset.loader

In [6]:
for i in range(2783, 2787):
    seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_val.__getitem__(i)
    print('Index', i)
    print(f"seq_x shape: {seq_x.shape}, seq_y shape: {seq_y.shape}, seq_x_mark shape: {seq_x_mark.shape}, seq_y_mark shape: {seq_y_mark.shape}")




NameError: name 'dataset_val' is not defined

### Checking if data is the same 

In [ ]:
for i in range(5):
    seq_x1, seq_y1, seq_x_mark1, seq_y_mark1 = dataset_val.__getitem__(i)  
    seq_x2, seq_y2, seq_x_mark2, seq_y_mark2 = dataset_loader1.__getitem__(i)

    # Check if the data is the same
    assert (seq_x1 == seq_x2).all(), f"seq_x1 and seq_x2 are not the same at index {i}"
    assert (seq_y1 == seq_y2).all(), f"seq_y1 and seq_y2 are not the same at index {i}"
    assert (seq_x_mark1 == seq_x_mark2).all(), f"seq_x_mark1 and seq_x_mark2 are not the same at index {i}"
    assert (seq_y_mark1 == seq_y_mark2).all(), f"seq_y_mark1 and seq_y_mark2 are not the same at index {i}"
    print(f"Data at index {i} is the same in both datasets.")

# NOTE THAT ETTH does not use the normalized timestamps

Data at index 0 is the same in both datasets.
Data at index 1 is the same in both datasets.
Data at index 2 is the same in both datasets.
Data at index 3 is the same in both datasets.
Data at index 4 is the same in both datasets.


In [ ]:
# for i in range(2875, 2885):
#     seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_val.__getitem__(i)
#     print('Index', i)
#     print(f"seq_x shape: {seq_x.shape}, seq_y shape: {seq_y.shape}, seq_x_mark shape: {seq_x_mark.shape}, seq_y_mark shape: {seq_y_mark.shape}")



In [9]:
import hydra
data_name = 'railway'

with hydra.initialize(
        version_base="1.3",
        config_path='configs',
        job_name="run"
    ):
    #print('Current config file: ', config_file)
    parameters = hydra.compose(
        config_name="run.yaml",
        overrides=[f'dataset={data_name}'], 
        return_hydra_config=True, 
    )
    dataset_loader2 = hydra.utils.instantiate(parameters.dataset.loader)


In [10]:
from lmetk.data.datasets import TimeSeriesDataset
data_splitter_params = parameters.dataset.loader.data_splitter
seq_len = data_splitter_params.seq_len
label_len = data_splitter_params.label_len
pred_len = data_splitter_params.pred_len

# get data
splitted_data, splitted_timestamps = dataset_loader2.get_data()

dataset_train = TimeSeriesDataset(data=splitted_data['train'], 
                  time_features=splitted_timestamps['train'], 
                  seq_len=seq_len, 
                  label_len=label_len, 
                  pred_len=pred_len)

dataset_val = TimeSeriesDataset(data=splitted_data['val'],
                    time_features=splitted_timestamps['val'], 
                    seq_len=seq_len, 
                    label_len=label_len, 
                    pred_len=pred_len)
dataset_test = TimeSeriesDataset(data=splitted_data['test'],
                    time_features=splitted_timestamps['test'], 
                    seq_len=seq_len, 
                    label_len=label_len, 
                    pred_len=pred_len)
print(f"Train dataset size: {dataset_train.__len__()}")
print(f"Validation dataset size: {dataset_val.__len__()}")
print(f"Test dataset size: {dataset_test.__len__()}")

Train dataset size: 37513
Validation dataset size: 4129
Test dataset size: 8113


In [15]:
seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_test.__getitem__(seq_len)

In [16]:
from lmetk.data.utils import reconstruct_datetime_from_features
reconstruct_datetime_from_features(seq_x_mark, ["year", "month", "day", "weekday", "hour"])

0     2023-01-24 00:00:00
1     2023-01-24 01:00:00
2     2023-01-24 02:00:00
3     2023-01-24 03:00:00
4     2023-01-24 04:00:00
              ...        
379   2023-02-08 19:00:00
380   2023-02-08 20:00:00
381   2023-02-08 21:00:00
382   2023-02-08 22:00:00
383   2023-02-08 23:00:00
Length: 384, dtype: datetime64[ns]

In [17]:
seq_x, seq_y, seq_x_mark, seq_y_mark = dataset_val.__getitem__(seq_len)
from lmetk.data.utils import reconstruct_datetime_from_features
reconstruct_datetime_from_features(seq_x_mark, ["year", "month", "day", "weekday", "hour"])

0     2022-08-02 00:00:00
1     2022-08-02 01:00:00
2     2022-08-02 02:00:00
3     2022-08-02 03:00:00
4     2022-08-02 04:00:00
              ...        
379   2022-08-17 19:00:00
380   2022-08-17 20:00:00
381   2022-08-17 21:00:00
382   2022-08-17 22:00:00
383   2022-08-17 23:00:00
Length: 384, dtype: datetime64[ns]

In [14]:
"2023-01-24 00:00:00"

'2023-01-24 00:00:00'